In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
# Change this for each of your 4 notebooks
MODEL_TYPE = "ContrastiveNet"  # Options: "MetaCNNLSTM", "DeepCNNLSTM", "TST", "ContrastiveNet"

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs"
FILE_NAME = f"1s3w_mamlpp100_{MODEL_TYPE}_2fcv_hpo"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: 1s3w_mamlpp100_ContrastiveNet_2fcv_hpo
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\1s3w_mamlpp100_ContrastiveNet_2fcv_hpo.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'mamlpp_pretr_ContrastiveNet_2fcv_hpo'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 106 trials.
Best value (Accuracy): 0.8854
Best Hyperparameters:
  meta_batchsize: 16
  maml_inner_steps: 5
  maml_inner_steps_eval: 15
  maml_alpha_init: 0.00612445284529773
  maml_alpha_init_eval: 0.008878205650763341
  outer_lr: 2.7341970583448306e-05
  wd: 1.3632627995562478e-06
  groupnorm_num_groups: 8
  use_GlobalAvgPooling: False
  finetuning_approach: full
  best_or_last_pretr: best
  context_emb_dim: 64
  context_pool_type: attn
  episodes_per_epoch_train: 500
  optimizer: adam
  use_maml_msl: hybrid
  maml_msl_num_epochs: 29


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL
params_to_plot = ["best_or_last_pretr", "context_emb_dim", "context_pool_type", "episodes_per_epoch_train", "finetuning_approach", "groupnorm_num_groups", 
                  "maml_inner_steps", "maml_inner_steps_eval", "maml_msl_num_epochs", "meta_batchsize", "optimizer", "use_GlobalAvgPooling", "use_maml_msl", "wd"]

# PRETRAINING STUFF / MISC
# Apparently finetuning_approach and use_maml_msl only had 1 value? Yes for finetuning currently, but use_maml_msl should have had true and false as well no? ...
params_to_plot_MISC = ["best_or_last_pretr", "finetuning_approach", "use_maml_msl"]

# ARCHITECTURE? --> What does this even mean? How can this vary if we are using the pretrained model....
params_to_plot_ARCH = ["context_emb_dim", "context_pool_type", "groupnorm_num_groups", "use_GlobalAvgPooling"]

# MAML++
params_to_plot_MAMLPP = ["maml_inner_steps", "maml_inner_steps_eval", "maml_msl_num_epochs", "use_maml_msl"]

# LRs MAML++
params_to_plot_LRS = ["maml_alpha_init", "maml_alpha_init_eval", "outer_lr"]

# Learning HPs
params_to_plot_HPS = ["episodes_per_epoch_train", "meta_batchsize", "optimizer", "wd"]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_MISC)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_ARCH)
fig_slice.show()

In [12]:
fig_slice = plot_slice(study, params=params_to_plot_MAMLPP)
fig_slice.show()

In [13]:
fig_slice = plot_slice(study, params=params_to_plot_LRS)
fig_slice.show()

In [14]:
fig_slice = plot_slice(study, params=params_to_plot_HPS)
fig_slice.show()

In [15]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
99,99,0.885417,0.889583,2026-03-26 13:13:56.669066,0 days 00:30:19.826240
80,80,0.880208,0.895312,2026-03-26 09:36:05.577165,0 days 00:35:20.154281
94,94,0.878646,0.899479,2026-03-26 12:03:06.840613,0 days 00:31:05.724267
96,96,0.878125,0.887500,2026-03-26 12:43:30.553567,0 days 00:30:11.857253
41,41,0.877604,0.872917,2026-03-26 02:13:34.665210,0 days 00:22:36.820889
101,101,0.876042,0.892188,2026-03-26 13:38:33.314188,0 days 00:31:31.968803
30,30,0.874479,0.885938,2026-03-26 00:01:23.486545,0 days 00:21:12.453784
98,98,0.873438,0.890104,2026-03-26 13:06:22.549786,0 days 00:31:59.015446
29,29,0.873438,0.877083,2026-03-25 23:50:43.135940,0 days 00:20:12.200766
79,79,0.873438,0.884896,2026-03-26 09:28:59.890219,0 days 00:36:34.245665
